# Week 9: PyTorch CNN Baseline for EEG Decoding

Goal:
Build a leakage-safe PyTorch training pipeline for motor imagery EEG classification.

Pipeline:

Processed EEG
→ PyTorch Dataset
→ DataLoader
→ CNN
→ CrossEntropyLoss
→ Backpropagation
→ Validation
→ Balanced Accuracy

## Imports and paths

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from neurosignallab.preprocessing import preprocess_subject


In [2]:

DATA_DIR = PROJECT_ROOT / "data"

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.14.0
Device: mps


### Create processed-data directory

In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR

PosixPath('/Users/heihei/Desktop/neurosignallab/data/processed')

### Save preprocessed data as arrays to drive

In [4]:
X_parts = []
y_parts = []
subject_parts = []
run_parts = []

for subject in range(1, 110):
    X_subject, y_subject, metadata_subject = preprocess_subject(
        subject=subject,
        data_dir=DATA_DIR
    )

    X_parts.append(
        X_subject.astype(np.float32, copy=False) #float32 for pytorch tensors
    )

    y_parts.append(
        y_subject.astype(np.int64, copy=False) #int64 for crossentropyloss to classify
    )

    subject_parts.append(
        metadata_subject["subject"].to_numpy(dtype=np.int16) #int16 to save memory
    )

    run_parts.append(
        metadata_subject["run"].to_numpy(dtype=np.int16)
    )

X_processed = np.concatenate(X_parts, axis=0)
y_processed = np.concatenate(y_parts)
subjects_processed = np.concatenate(subject_parts)
runs_processed = np.concatenate(run_parts)

del X_parts, y_parts, subject_parts, run_parts

Do you want to set the path:
    /Users/heihei/Desktop/neurosignallab/data
as the default EEGBCI dataset path in the mne-python config [y]/n?  y


Download complete in 02m48s (7.4 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m12s (7.3 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m52s (7.4 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S003/S003R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m54s (7.3 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S004/S004R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

...

Download complete in 01m13s (7.3 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S105/S105R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S105/S105R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S105/S105R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m16s (7.4 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S106/S106R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S106/S106R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S106/S106R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m12s (7.4 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S107/S107R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S107/S107R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S107/S107R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m19s (7.3 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S108/S108R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S108/S108R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S108/S108R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

Download complete in 01m39s (7.3 MB)
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S109/S109R04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S109/S109R08.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Extracting EDF parameters from /Users/heihei/Desktop/neurosignallab/data/MNE-eegbci-data/files/eegmmidb/1.0.0/S109/S109R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Used Annotations descriptions: [np.str_('T0'), np.str_('

### Verify dataset

In [5]:
print("X:", X_processed.shape, X_processed.dtype)
print("y:", y_processed.shape, y_processed.dtype)
print("subjects:", subjects_processed.shape, subjects_processed.dtype)
print("runs:", runs_processed.shape, runs_processed.dtype)

print()
print("Classes:", np.unique(y_processed, return_counts=True))
print("Number of subjects:", np.unique(subjects_processed).size)
print("Runs:", np.unique(runs_processed))

assert X_processed.shape == (4898, 64, 481)

assert y_processed.shape == (4898,)
assert subjects_processed.shape == (4898,)
assert runs_processed.shape == (4898,)

assert X_processed.dtype == np.float32
assert y_processed.dtype == np.int64

assert set(np.unique(y_processed)) == {0, 1}
assert np.unique(subjects_processed).size == 109
assert set(np.unique(runs_processed)) == {4, 8, 12}

print("NaN values:", np.isnan(X_processed).sum())
print("Infinite values:", np.isinf(X_processed).sum())

X: (4898, 64, 481) float32
y: (4898,) int64
subjects: (4898,) int16
runs: (4898,) int16

Classes: (array([0, 1]), array([2470, 2428]))
Number of subjects: 109
Runs: [ 4  8 12]
NaN values: 0
Infinite values: 0


### Save processed arrays

In [6]:
np.save(
    PROCESSED_DIR / "X_motor_imagery.npy",
    X_processed
)

np.save(
    PROCESSED_DIR / "y_motor_imagery.npy",
    y_processed
)

np.save(
    PROCESSED_DIR / "subjects.npy",
    subjects_processed
)

np.save(
    PROCESSED_DIR / "runs.npy",
    runs_processed
)

print("Saved processed EEG dataset to:")
print(PROCESSED_DIR)

Saved processed EEG dataset to:
/Users/heihei/Desktop/neurosignallab/data/processed


### Check if data can be loaded

In [7]:
X_check = np.load(
    PROCESSED_DIR / "X_motor_imagery.npy",
    mmap_mode="r" #Read only
)

y_check = np.load(
    PROCESSED_DIR / "y_motor_imagery.npy"
)

subjects_check = np.load(
    PROCESSED_DIR / "subjects.npy"
)

runs_check = np.load(
    PROCESSED_DIR / "runs.npy"
)

print(X_check.shape, X_check.dtype)
print(y_check.shape, y_check.dtype)
print(subjects_check.shape)
print(runs_check.shape)

(4898, 64, 481) float32
(4898,) int64
(4898,)
(4898,)


## Subject-disjoint train / validation / test split

Subjects are split into training, validation,
and test sets.

This prevents the same person's EEG from appearing in more than one set.
The test subjects remain untouched during model development.

In [8]:
X = X_check
y = y_check
subjects = subjects_check
runs = runs_check

unique_subjects = np.unique(subjects)

split_rng = np.random.default_rng(SEED)
shuffled_subjects = split_rng.permutation(unique_subjects)

n_val = 16
n_test = 17

val_subjects = np.sort(shuffled_subjects[:n_val])
test_subjects = np.sort(shuffled_subjects[n_val:n_val + n_test])
train_subjects = np.sort(shuffled_subjects[n_val + n_test:])

train_mask = np.isin(subjects, train_subjects)
val_mask = np.isin(subjects, val_subjects)
test_mask = np.isin(subjects, test_subjects)

print("Subjects")
print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

print("\nTrials")
print("Train:", train_mask.sum())
print("Validation:", val_mask.sum())
print("Test:", test_mask.sum())

Subjects
Train: 76
Validation: 16
Test: 17

Trials
Train: 3423
Validation: 718
Test: 757


### Check

In [9]:
assert set(train_subjects).isdisjoint(val_subjects)
assert set(train_subjects).isdisjoint(test_subjects)
assert set(val_subjects).isdisjoint(test_subjects)

assert (
    train_mask.sum()
    + val_mask.sum()
    + test_mask.sum()
    == len(y)
)

print("No subject overlap.")
print("Every trial belongs to exactly one split.")

No subject overlap.
Every trial belongs to exactly one split.


### Save determined splits in csv

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

split_table = pd.DataFrame({
    "subject": np.concatenate([
        train_subjects,
        val_subjects,
        test_subjects
    ]),
    "split": (
        ["train"] * len(train_subjects)
        + ["validation"] * len(val_subjects)
        + ["test"] * len(test_subjects)
    )
}).sort_values("subject")

split_table.to_csv(
    RESULTS_DIR / "week9_subject_split.csv",
    index=False
)

split_table.head(10)

## PyTorch Dataset

The Dataset maps an integer index to one EEG trial and its class label.

The stored EEG remains in volts on disk. Each trial is converted to
`float32` and scaled to microvolts when retrieved.

The Dataset stores trial indices rather than copying the complete
training and validation arrays into memory.

### Create indices

In [10]:
train_indices = np.flatnonzero(train_mask)
val_indices = np.flatnonzero(val_mask)
test_indices = np.flatnonzero(test_mask)

### Create pytorch dataset class

In [11]:
class EEGDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        trial_index = self.indices[index]

        x = torch.tensor(
            self.X[trial_index],
            dtype=torch.float32
        ) * 1e6

        y = torch.tensor(
            self.y[trial_index],
            dtype=torch.long
        )

        return x, y

### Instate dataset

In [12]:
train_dataset = EEGDataset(
    X,
    y,
    train_indices
)

val_dataset = EEGDataset(
    X,
    y,
    val_indices
)

### Verify dataset

In [13]:
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

x_example, y_example = train_dataset[0]

print("\nOne EEG trial")
print("Shape:", x_example.shape)
print("dtype:", x_example.dtype)

print("\nLabel")
print("Value:", y_example.item())
print("dtype:", y_example.dtype)

Training examples: 3423
Validation examples: 718

One EEG trial
Shape: torch.Size([64, 481])
dtype: torch.float32

Label
Value: 0
dtype: torch.int64


## DataLoaders

The DataLoader groups individual EEG trials from the Dataset into
mini-batches for neural-network training.

Training examples are shuffled between epochs, while validation examples
are kept in a fixed order.

Shuffling changes only the order of trials inside the training set.
It does not move subjects between training and validation.

### Load data into batches and shuffle training set

In [14]:
BATCH_SIZE = 32

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

### Inspect batch

In [15]:
X_batch, y_batch = next(iter(train_loader))

print("EEG batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)

print("EEG dtype:", X_batch.dtype)
print("Label dtype:", y_batch.dtype)

print("Labels:", y_batch)

EEG batch shape: torch.Size([32, 64, 481])
Label batch shape: torch.Size([32])
EEG dtype: torch.float32
Label dtype: torch.int64
Labels: tensor([0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0,
        1, 1, 1, 0, 1, 1, 0, 1])


## Simple CNN

The first neural network is deliberately simple.

Its purpose is to verify the complete PyTorch training pipeline before
introducing a more specialized EEG architecture such as EEGNet.

Input shape:

`(batch, 64 channels, 481 time samples)`

Architecture:

EEG
→ temporal convolution (Output 16 feature maps using 64 channels and 16 learned filters from the convid function)
→ ReLU (introduce nonlinear classifier by removing negative values)
→ temporal pooling (Halve temporal dimension)
→ convolution (Produce 32 new feature maps by summing old ones)
→ ReLU (adds another non-linear layer)
→ temporal pooling (Halve temporal dimension)
→ flatten (Takes 32 features and 120 values 3840 learned values)
→ linear classifier (Match the features to which class it is)
→ 2 logits (Each class gets weighted by every learned value)

### Create smple EEGCNN architecture

In [18]:
class SimpleEEGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(
                in_channels=64,
                out_channels=16,
                kernel_size=15,
                padding=7
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=16,
                out_channels=32,
                kernel_size=15,
                padding=7
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )

        self.classifier = nn.Linear(
            32 * 120,
            2
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)

        return x

### Test simple EEGCNN

In [20]:
model = SimpleEEGCNN().to(device)

X_batch_device = X_batch.to(device)
y_batch_device = y_batch.to(device)

with torch.no_grad():
    logits = model(X_batch_device)

print("Input shape:", X_batch_device.shape)
print("Output shape:", logits.shape)
print("Device:", logits.device)

print("\nFirst trial logits:")
print(logits[0])

Input shape: torch.Size([32, 64, 481])
Output shape: torch.Size([32, 2])
Device: mps:0

First trial logits:
tensor([2.5401, 0.7921], device='mps:0')


### Connect network to CrossEntropyLoss

In [21]:
criterion = nn.CrossEntropyLoss()

logits = model(X_batch_device)

loss = criterion(
    logits,
    y_batch_device
)

print("Loss:", loss.item())

Loss: 1.1855854988098145


## Backpropagation

CrossEntropyLoss measures how wrong the current predictions are.

Backpropagation then calculates how each learnable parameter contributed
to that loss.

These gradients do not change the weights by themselves. They tell the
optimizer how the weights should be adjusted.

### Create optimizer and backpropogate

In [22]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

optimizer.zero_grad()

logits = model(X_batch_device)
loss = criterion(logits, y_batch_device)

loss.backward()

### Inspect gradient

In [23]:
first_conv_grad = model.features[0].weight.grad

print("Gradient shape:", first_conv_grad.shape)
print("Mean absolute gradient:", first_conv_grad.abs().mean().item())

Gradient shape: torch.Size([16, 64, 15])
Mean absolute gradient: 0.12227430194616318


### Check optimizing function

In [24]:
loss_before = loss.item()

optimizer.step()

with torch.no_grad():
    logits_after = model(X_batch_device)
    loss_after = criterion(logits_after, y_batch_device).item()

print("Loss before update:", loss_before)
print("Loss after update:", loss_after)

Loss before update: 1.1855854988098145
Loss after update: 8.118551254272461


### Overfit model to check learning

In [25]:
model = SimpleEEGCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

for step in range(300):
    optimizer.zero_grad()

    logits = model(X_batch_device)
    loss = criterion(logits, y_batch_device)

    loss.backward()
    optimizer.step()

    if step in {0, 49, 99, 199, 299}:
        accuracy = (
            logits.argmax(dim=1) == y_batch_device
        ).float().mean().item()

        print(
            f"Step {step + 1:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Accuracy: {accuracy:.3f}"
        )

Step   1 | Loss: 2.0006 | Accuracy: 0.406
Step  50 | Loss: 0.0290 | Accuracy: 1.000
Step 100 | Loss: 0.0101 | Accuracy: 1.000
Step 200 | Loss: 0.0034 | Accuracy: 1.000
Step 300 | Loss: 0.0017 | Accuracy: 1.000


## Conclusion

The PyTorch baseline pipeline is functioning correctly.

- Preprocessed EEG data were loaded successfully.
- A fixed subject-level train/validation/test split was created.
- PyTorch Dataset and DataLoader objects were constructed.
- The baseline 1D CNN produced logits with the expected `(B, 2)` shape.
- CrossEntropyLoss was successfully connected to the raw logits.
- Backpropagation produced non-zero gradients through the convolutional layers.
- Adam successfully updated the network parameters.
- A 32-trial tiny-batch overfitting test reached 100% training accuracy and near-zero loss, confirming that the model and training pipeline can learn.

Next: train the baseline CNN on the full training set and evaluate development performance on the validation subjects.
    